# Transformers y mecanismo de atención

La arquitectura Transformer (2017) es la base de todos los LLMs modernos.
Implementamos atención escalada, multi-cabeza y codificación posicional desde NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('NumPy', np.__version__)

## 1. Embeddings — palabras como vectores

In [ ]:
DIMENSION = 8
VOCABULARIO = {
    'contrato': 0, 'cláusula': 1, 'precio': 2, 'vencimiento': 3,
    'legal': 4, 'empresa': 5, 'proveedor': 6, 'cliente': 7,
    'tecnología': 8, 'software': 9, 'API': 10, 'datos': 11,
}

np.random.seed(42)
matriz = np.random.randn(len(VOCABULARIO), DIMENSION)
matriz = matriz / np.linalg.norm(matriz, axis=1, keepdims=True)

def similitud(p1: str, p2: str) -> float:
    return float(np.dot(matriz[VOCABULARIO[p1]], matriz[VOCABULARIO[p2]]))

pares = [
    ('contrato', 'cláusula'),
    ('contrato', 'precio'),
    ('empresa', 'proveedor'),
    ('legal', 'cláusula'),
]
print('Similitudes (embeddings aleatorios — sin entrenar):')
for p1, p2 in pares:
    print(f'  {p1} ↔ {p2}: {similitud(p1, p2):.3f}')

## 2. Scaled Dot-Product Attention

In [ ]:
def atencion(Q: np.ndarray, K: np.ndarray, V: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Scaled Dot-Product Attention. Q, K, V: [n_tokens, d_modelo]"""
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    exp_s = np.exp(scores - scores.max(axis=-1, keepdims=True))
    pesos = exp_s / exp_s.sum(axis=-1, keepdims=True)
    return pesos @ V, pesos


np.random.seed(0)
n_tokens, d_modelo = 4, 6
Q = np.random.randn(n_tokens, d_modelo)
K = np.random.randn(n_tokens, d_modelo)
V = np.random.randn(n_tokens, d_modelo)

output, pesos = atencion(Q, K, V)

tokens = ['el', 'contrato', 'legal', 'vence']
print('Pesos de atención (fila i → cuánta atención presta el token i a cada token):')
print(f'{"":8}', end='')
for t in tokens:
    print(f'{t:>10}', end='')
print()
for i, fila in enumerate(pesos):
    print(f'{tokens[i]:8}', end='')
    for p in fila:
        print(f'{p:>10.3f}', end='')
    print()

print(f'\nOutput shape: {output.shape}  (igual que input)')

## 3. Visualizar los pesos de atención

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(pesos, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(n_tokens))
ax.set_yticks(range(n_tokens))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_xlabel('Token clave (key)')
ax.set_ylabel('Token consulta (query)')
ax.set_title('Mapa de atención')
plt.colorbar(im, ax=ax)

for i in range(n_tokens):
    for j in range(n_tokens):
        ax.text(j, i, f'{pesos[i,j]:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Atención multi-cabeza

In [ ]:
class AtencionMultiCabeza:
    def __init__(self, d_modelo: int, n_cabezas: int):
        assert d_modelo % n_cabezas == 0
        self.d_k = d_modelo // n_cabezas
        self.n_cabezas = n_cabezas
        np.random.seed(42)
        self.W_Q = np.random.randn(d_modelo, d_modelo) * 0.1
        self.W_K = np.random.randn(d_modelo, d_modelo) * 0.1
        self.W_V = np.random.randn(d_modelo, d_modelo) * 0.1
        self.W_O = np.random.randn(d_modelo, d_modelo) * 0.1

    def _atencion(self, Q, K, V):
        scores = Q @ K.T / np.sqrt(self.d_k)
        exp_s = np.exp(scores - scores.max(axis=-1, keepdims=True))
        pesos = exp_s / exp_s.sum(axis=-1, keepdims=True)
        return pesos @ V, pesos

    def forward(self, x: np.ndarray) -> tuple[np.ndarray, list]:
        Q = x @ self.W_Q
        K = x @ self.W_K
        V = x @ self.W_V
        outputs, todos_pesos = [], []
        for i in range(self.n_cabezas):
            s, e = i * self.d_k, (i + 1) * self.d_k
            o, p = self._atencion(Q[:, s:e], K[:, s:e], V[:, s:e])
            outputs.append(o)
            todos_pesos.append(p)
        return np.concatenate(outputs, axis=-1) @ self.W_O, todos_pesos


n_tokens, d_modelo, n_cabezas = 6, 8, 2
x = np.random.randn(n_tokens, d_modelo)
mha = AtencionMultiCabeza(d_modelo=d_modelo, n_cabezas=n_cabezas)
output, pesos = mha.forward(x)

print(f'Entrada : [{n_tokens}, {d_modelo}]')
print(f'Salida  : {output.shape}')
print(f'Cabezas : {n_cabezas} × d_k={d_modelo // n_cabezas}')
print(f'\nCabeza 1 — pesos token 0: {pesos[0][0].round(3)}')
print(f'Cabeza 2 — pesos token 0: {pesos[1][0].round(3)}')
print('(Diferentes pesos = cada cabeza aprende un aspecto distinto del texto)')

## 5. Codificación posicional sinusoidal

In [ ]:
def positional_encoding(n_tokens: int, d_modelo: int) -> np.ndarray:
    PE = np.zeros((n_tokens, d_modelo))
    pos = np.arange(n_tokens)[:, np.newaxis]
    div = np.power(10000, np.arange(0, d_modelo, 2) / d_modelo)
    PE[:, 0::2] = np.sin(pos / div)
    PE[:, 1::2] = np.cos(pos / div)
    return PE

PE = positional_encoding(50, 64)

plt.figure(figsize=(10, 4))
plt.pcolormesh(PE.T, cmap='RdBu')
plt.colorbar()
plt.xlabel('Posición del token')
plt.ylabel('Dimensión del embedding')
plt.title('Codificación posicional sinusoidal')
plt.tight_layout()
plt.show()

print('Cada posición tiene un "fingerprint" único gracias a las frecuencias sinusoidales.')
print('Modelos modernos (LLaMA, Claude) usan RoPE, que generaliza a secuencias más largas.')

## 6. Generación autoregresiva

In [ ]:
import random

def ilustrar_generacion(prompt: str, n_tokens: int = 5) -> list[str]:
    """Ilustra cómo un LLM genera tokens uno a uno (tokens aleatorios como demo)."""
    vocabulario = ['contrato', 'válido', 'hasta', 'fecha', 'límite',
                   'el', 'la', 'es', 'de', 'en', 'y', 'con', '.']
    random.seed(42)
    tokens = prompt.split()
    for paso in range(n_tokens):
        nuevo = random.choice(vocabulario)
        print(f'Paso {paso+1}: "{" ".join(tokens)}" → genera "{nuevo}"')
        tokens.append(nuevo)
    return tokens

print('Generación autoregresiva (tokens aleatorios — demo):')
tokens = ilustrar_generacion('El contrato')
print(f'\nTexto completo: "{" ".join(tokens)}"')

## 7. Ley de escalado de Chinchilla

In [ ]:
def perdida_estimada(n_params: float, n_tokens_train: float) -> float:
    """Aproximación simplificada de la ley de Chinchilla (DeepMind, 2022)."""
    A, alpha = 406.4, 0.34
    B, beta  = 410.7, 0.28
    C = 1.69
    return A / (n_params ** alpha) + B / (n_tokens_train ** beta) + C

configs = [
    ('GPT-2 (1.5B)',            1.5e9,  300e9),
    ('GPT-3 (175B)',            175e9,  300e9),
    ('LLaMA-2-7B',              7e9,    2e12),
    ('Óptimo 70B (Chinchilla)', 70e9,   1.4e12),
]

print(f'{"Modelo":<30} {"Parámetros":>12} {"Tokens train":>14} {"Pérdida est.":>13}')
print('-' * 72)
for nombre, n_p, n_t in configs:
    p_str = f'{n_p/1e9:.0f}B'
    t_str = f'{n_t/1e12:.1f}T' if n_t >= 1e12 else f'{n_t/1e9:.0f}B'
    print(f'{nombre:<30} {p_str:>12} {t_str:>14} {perdida_estimada(n_p, n_t):>13.3f}')

print('\nRegla Chinchilla: escalar parámetros y datos por igual.')
print('Ratio óptimo: ~20 tokens de entrenamiento por parámetro.')